In [10]:
from pathlib import Path
import json
import re
import pandas as pd
from json_repair import repair_json

In [11]:
run_name = "04022026-all-screening-04"

# Resolve artifacts directory relative to the repo root.
ARTIFACTS_DIR = (Path('labelling/artifacts') if Path('labelling/artifacts').exists() else (Path.cwd() / 'labelling' / 'artifacts')).resolve()
batches_dir = ARTIFACTS_DIR / 'batches'
outputs_dir = ARTIFACTS_DIR / 'batch_outputs'

if not (batches_dir / run_name).exists():
    raise FileNotFoundError(f"Run {run_name!r} not found under {batches_dir}")


In [12]:
def read_manifest(run: str) -> dict:
    path = batches_dir / run / "manifest.json"
    with open(path) as f:
        return json.load(f)


def load_batch_records(run: str, keep_cols=("UT", "title", "abstract", "doi")) -> pd.DataFrame:
    data_dir = batches_dir / run / "data"
    frames = []
    if not data_dir.exists():
        return pd.DataFrame(columns=keep_cols)
    for fp in sorted(data_dir.glob("*.jsonl")):
        frames.append(pd.read_json(fp, lines=True))
    if not frames:
        return pd.DataFrame(columns=keep_cols)
    df = pd.concat(frames, ignore_index=True)
    cols = [c for c in keep_cols if c in df.columns]
    return df[cols]


def parse_json_text(text: str):
    if not text:
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    if repair_json:
        try:
            return repair_json(text, return_objects=True)
        except Exception:
            pass

    cleaned = re.sub(r",\s*([}\]])", r"\1", text.replace("\r", "").replace("\n", " "))
    try:
        return json.loads(cleaned)
    except Exception:
        return None


ALLOWED_REASON_KEYS = {"stressor_spans", "use_type", "evidence_span", "biodiversity_span", "link_span"}


def extract_output_payload(output_list):
    # Return the first JSON payload found in the model output content, else None.
    for item in output_list or []:
        content_blocks = item.get("content") or []
        for block in content_blocks:
            text = block.get("text") if isinstance(block, dict) else None
            if not text:
                continue
            parsed = parse_json_text(text)
            if parsed is not None:
                return parsed
    return None


def flatten_steps(parsed: dict) -> dict:
    flat = {}
    if not isinstance(parsed, dict):
        return flat
    for step_key, step_val in parsed.items():
        if not isinstance(step_val, dict):
            continue
        flat[f"{step_key}_label"] = step_val.get("label")
        reason = step_val.get("reason")
        flat[f"{step_key}_reason"] = reason
        if isinstance(reason, dict):
            for k, v in reason.items():
                if ALLOWED_REASON_KEYS and k not in ALLOWED_REASON_KEYS:
                    continue
                flat[f"{step_key}_reason_{k}"] = v
    return flat


def parse_output_record(rec: dict) -> dict:
    body = (rec.get("response") or {}).get("body") or {}
    model = body.get("model")
    created_at = body.get("created_at")
    parsed = extract_output_payload(body.get("output"))
    flat = flatten_steps(parsed)
    return {
        "custom_id": rec.get("custom_id"),
        "model": model,
        "created_at": created_at,
        "raw_output": parsed,
        **flat,
    }


def load_outputs(run: str) -> pd.DataFrame:
    out_dir = outputs_dir / run
    if not out_dir.exists():
        return pd.DataFrame()
    files = sorted(out_dir.glob("*.jsonl"))
    rows = []
    for fp in files:
        with open(fp) as f:
            for line in f:
                rec = json.loads(line)
                rows.append(parse_output_record(rec))
    df = pd.DataFrame(rows)
    if not df.empty and "created_at" in df.columns:
        df["created_at"] = pd.to_datetime(df["created_at"], unit="s", utc=True)
    return df


In [13]:
# --- Load the selected run ---
manifest = read_manifest(run_name)
batch_df = load_batch_records(run_name)
outputs_df = load_outputs(run_name)

# Join on UT/custom_id to align documents with model outputs
joined = batch_df.merge(outputs_df, left_on="UT", right_on="custom_id", how="left")

print(f"Run: {manifest['run_name']}")
print(f"Documents: {len(batch_df):,}")
print(f"Output rows: {len(outputs_df):,}")

Run: 04022026-all-screening-04
Documents: 4,495
Output rows: 4,495


In [14]:
# --- Derived prediction columns ---
for col in ["step_0_label", "step_1_label", "step_2_label", "step_3_label"]:
    if col in joined.columns:
        joined[col] = pd.to_numeric(joined[col], errors="coerce").fillna(0).astype(int)
    else:
        joined[col] = 0

joined["label_1_3"] = joined[["step_1_label", "step_2_label", "step_3_label"]].sum(axis=1)
joined["pred_screening"] = joined.apply(
    lambda r: '["ELIGIBLE"]' if (r["step_0_label"] == 0 and r["label_1_3"] == 3) else '["NOT_ELIGIBLE"]',
    axis=1,
)


In [15]:
# --- Load curated true labels and join ---
def load_true_labels():
    folder = Path('data/labels/l0-batched-curated')
    frames = []
    for fp in folder.glob('*.csv'):
        frames.append(pd.read_csv(fp))
    if not frames:
        return pd.DataFrame(columns=['ut_unique_wos_id_', 'true_label'])
    df = pd.concat(frames, ignore_index=True)
    label_col = 'eligbility' if 'eligbility' in df.columns else ('eligibility' if 'eligibility' in df.columns else None)
    if label_col is None:
        return pd.DataFrame(columns=['ut_unique_wos_id_', 'true_label'])
    df = df.rename(columns={label_col: 'true_label'})
    return df[['ut_unique_wos_id_', 'true_label']].drop_duplicates()

truth_df = load_true_labels()
joined = joined.merge(truth_df, left_on='custom_id', right_on='ut_unique_wos_id_', how='left')


In [16]:
# --- Confusion matrix (ELIGIBLE treated as positive) and per-row flags ---
eligible_token = '["ELIGIBLE"]'
pred_label = joined['pred_screening'].fillna('').astype(str).str.strip()
true_label = joined['true_label'].fillna('').astype(str).str.strip()
has_truth = joined['true_label'].notna()

pred_pos = pred_label == eligible_token
true_pos = true_label == eligible_token

for col in ['tp', 'fp', 'fn', 'tn']:
    joined[col] = pd.NA

joined.loc[has_truth, 'tp'] = (pred_pos & true_pos & has_truth).astype('Int64')
joined.loc[has_truth, 'fp'] = (pred_pos & ~true_pos & has_truth).astype('Int64')
joined.loc[has_truth, 'fn'] = (~pred_pos & true_pos & has_truth).astype('Int64')
joined.loc[has_truth, 'tn'] = (~pred_pos & ~true_pos & has_truth).astype('Int64')

tp = joined['tp'].fillna(0).sum()
fp = joined['fp'].fillna(0).sum()
fn = joined['fn'].fillna(0).sum()
tn = joined['tn'].fillna(0).sum()

confusion = pd.DataFrame({
    'pred_ELIGIBLE': [tp, fp],
    'pred_NOT_ELIGIBLE': [fn, tn],
}, index=['true_ELIGIBLE', 'true_NOT_ELIGIBLE'])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
metrics = pd.DataFrame({
    'metric': ['precision', 'recall', 'f1'],
    'value': [precision, recall, f1],
})
confusion, metrics


/var/folders/sf/ksfgwr354pl2gjdb_f0yn3vr0000gn/T/ipykernel_11752/1518559525.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tp = joined['tp'].fillna(0).sum()
/var/folders/sf/ksfgwr354pl2gjdb_f0yn3vr0000gn/T/ipykernel_11752/1518559525.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fp = joined['fp'].fillna(0).sum()
/var/folders/sf/ksfgwr354pl2gjdb_f0yn3vr0000gn/T/ipykernel_11752/1518559525.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

(                   pred_ELIGIBLE  pred_NOT_ELIGIBLE
 true_ELIGIBLE               2560                346
 true_NOT_ELIGIBLE            551               1038,
       metric     value
 0  precision  0.822887
 1     recall  0.880936
 2         f1  0.850922)

In [17]:
# --- Quick inspection helpers ---
# Show a single record with parsed reasons
example = joined[joined.UT == "WOS:A1996VZ68900009"].iloc[0]
example[[c for c in joined.columns if c not in {"abstract"}]]

UT                                                               WOS:A1996VZ68900009
title                              Biocoenotic changes of the zooplankton in the ...
doi                                                           10.1006/jmsc.1996.0117
custom_id                                                        WOS:A1996VZ68900009
model                                                          gpt-5-nano-2025-08-07
created_at                                                 2026-02-05 17:52:12+00:00
raw_output                         {'step_0': {'label': 0, 'reason': None}, 'step...
step_0_label                                                                       0
step_0_reason                                                                   None
step_1_label                                                                       1
step_1_reason                      {'stressor_spans': ['nutrient availability', '...
step_1_reason_stressor_spans               [nutrient availability

In [9]:
joined.to_csv(f"screening.csv", index=False)